In [ ]:
from __future__ import annotations

import inspect
import importlib.util
import json
import os
import random
import re
import subprocess
import sys
import zipfile
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple


if "KAGGLE_KERNEL_RUN_TYPE" not in os.environ:
    raise EnvironmentError(
        "Run this notebook on Kaggle with GPU enabled. It is not intended for local execution."
    )


BASE_REQUIRED_PACKAGES = {
    "kagglehub": "kagglehub>=1.0.0",
    "datasets": "datasets>=3.0.0",
    "peft": "peft>=0.15.0",
    "bitsandbytes": "bitsandbytes>=0.46.0",
    "accelerate": "accelerate>=1.5.0",
    "sentencepiece": "sentencepiece>=0.2.0",
    "transformers": "transformers>=4.57.3",
}

MODEL_RUNTIME_PACKAGES = {
    "mamba_ssm": "mamba-ssm[causal-conv1d]>=2.3.1",
    "causal_conv1d": "causal-conv1d>=1.6.1",
}

MINIMUM_CUDA_CAPABILITY = (7, 0)


def install_missing_packages(
    requirements: Dict[str, str],
    retry_with_no_build_isolation: bool = False,
) -> None:
    missing_packages = [
        package_name
        for module_name, package_name in requirements.items()
        if importlib.util.find_spec(module_name) is None
    ]

    if not missing_packages:
        return

    install_command = [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    try:
        subprocess.check_call(install_command)
    except subprocess.CalledProcessError:
        if not retry_with_no_build_isolation:
            raise

        retry_command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-build-isolation",
            *missing_packages,
        ]
        subprocess.check_call(retry_command)


install_missing_packages(BASE_REQUIRED_PACKAGES)


import kagglehub
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import display
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)


assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before running this notebook."


def ensure_supported_kaggle_gpu() -> Tuple[str, Tuple[int, int], float]:
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    gpu_capability = (int(major), int(minor))
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"GPU: {gpu_name} ({gpu_memory_gb:.1f} GB, sm_{gpu_capability[0]}{gpu_capability[1]})")

    if gpu_capability < MINIMUM_CUDA_CAPABILITY:
        raise RuntimeError(
            "Kaggle assigned an unsupported GPU for this Nemotron stack: "
            f"{gpu_name} (sm_{gpu_capability[0]}{gpu_capability[1]}). "
            "This notebook requires a Kaggle GPU with compute capability sm_70 or newer. "
            "Stop the Kaggle session, start a new GPU session, and rerun from the first cell until you get "
            "a T4, L4, A10G, or A100 instead of a P100."
        )

    return gpu_name, gpu_capability, gpu_memory_gb


gpu_name, gpu_capability, gpu_memory_gb = ensure_supported_kaggle_gpu()
install_missing_packages(MODEL_RUNTIME_PACKAGES, retry_with_no_build_isolation=True)

SEED = 42
COMPETITION_HANDLE = "nvidia-nemotron-model-reasoning-challenge"
MODEL_HANDLE = "metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1"
AUX_DATASET_HANDLE = "thedevastator/grade-school-math-8k-q-a"
ACTIVE_VARIANT = os.environ.get("NEMOTRON_VARIANT", "competition_plus_gsm8k")
RUN_VARIANTS = {
    "smoke_debug": {
        "debug": True,
        "use_aux_dataset": True,
        "max_competition_train_samples": 64,
        "max_aux_train_samples": 96,
        "max_valid_samples": 16,
        "num_train_epochs": 1,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "learning_rate": 3e-4,
        "lora_rank": 8,
        "max_seq_length": 1024,
        "logging_steps": 2,
        "preview_max_new_tokens": 128,
    },
    "competition_plus_gsm8k": {
        "debug": True,
        "use_aux_dataset": True,
        "max_competition_train_samples": 256,
        "max_aux_train_samples": 384,
        "max_valid_samples": 64,
        "num_train_epochs": 1,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2e-4,
        "lora_rank": 16,
        "max_seq_length": 1536,
        "logging_steps": 5,
        "preview_max_new_tokens": 192,
    },
    "higher_rank": {
        "debug": False,
        "use_aux_dataset": True,
        "max_competition_train_samples": 384,
        "max_aux_train_samples": 512,
        "max_valid_samples": 64,
        "num_train_epochs": 1,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "learning_rate": 1.5e-4,
        "lora_rank": 32,
        "max_seq_length": 2048,
        "logging_steps": 10,
        "preview_max_new_tokens": 256,
    },
}

if ACTIVE_VARIANT not in RUN_VARIANTS:
    raise KeyError(
        f"Unknown ACTIVE_VARIANT={ACTIVE_VARIANT!r}. Choose one of: {sorted(RUN_VARIANTS)}"
    )

VARIANT = RUN_VARIANTS[ACTIVE_VARIANT]
WORKING_DIR = Path("/kaggle/working")
RUN_NAME = f"nemotron_{ACTIVE_VARIANT}_seed{SEED}"
RUN_DIR = WORKING_DIR / RUN_NAME
ARTIFACT_DIR = RUN_DIR / "artifacts"
ADAPTER_DIR = ARTIFACT_DIR / "adapter"
SUBMISSION_PATH = RUN_DIR / "submission.zip"
BF16_AVAILABLE = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
COMPUTE_DTYPE = torch.bfloat16 if BF16_AVAILABLE else torch.float16
MAX_SEQ_LENGTH = int(VARIANT["max_seq_length"])
MAX_VALID_SAMPLES = int(VARIANT["max_valid_samples"])
NUM_TRAIN_EPOCHS = float(VARIANT["num_train_epochs"])
TRAIN_BATCH_SIZE = int(VARIANT["per_device_train_batch_size"])
GRADIENT_ACCUMULATION_STEPS = int(VARIANT["gradient_accumulation_steps"])
LEARNING_RATE = float(VARIANT["learning_rate"])
LORA_RANK = int(VARIANT["lora_rank"])
SYSTEM_PROMPT = (
    "You are solving a reasoning benchmark problem. Solve it carefully and end the visible response "
    "with exactly one final answer written as \\boxed{...}."
)

QUESTION_COLUMN_CANDIDATES = (
    "question",
    "problem",
    "prompt",
    "query",
    "task",
    "instruction",
    "input",
)
ANSWER_COLUMN_CANDIDATES = (
    "answer",
    "solution",
    "target",
    "output",
    "final_answer",
    "ground_truth",
    "label",
)
FINAL_MARKER_PATTERN = re.compile(r"####\s*([^\n]+)")
BOXED_PATTERN = re.compile(r"\\boxed\{([^{}]+)\}")
NUMBER_PATTERN = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?(?:/\d+)?")

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
    torch.backends.cuda.matmul.allow_tf32 = True

print(json.dumps({"active_variant": ACTIVE_VARIANT, "variant_config": VARIANT}, indent=2))

input_root = Path("/kaggle/input")
if input_root.exists():
    attached_inputs = sorted(child.name for child in input_root.iterdir())
    print(f"Attached Kaggle inputs: {attached_inputs}")



In [ ]:
def handle_slug(handle: str) -> str:
    if "/" not in handle:
        return handle
    return handle.split("/")[1]


def resolve_competition_dir(handle: str) -> Path:
    mounted_dir = Path("/kaggle/input") / handle
    if mounted_dir.exists():
        return mounted_dir
    return Path(kagglehub.competition_download(handle))


def resolve_dataset_dir(handle: str) -> Path:
    mounted_dir = Path("/kaggle/input") / handle_slug(handle)
    if mounted_dir.exists():
        return mounted_dir
    return Path(kagglehub.dataset_download(handle))


def resolve_model_dir(handle: str) -> Path:
    input_root = Path("/kaggle/input")
    model_slug = handle_slug(handle).lower()

    if input_root.exists():
        for config_path in input_root.rglob("config.json"):
            model_dir = config_path.parent
            model_dir_text = str(model_dir).lower()
            has_weights = model_dir.joinpath("model.safetensors.index.json").exists() or any(
                model_dir.glob("model-*.safetensors")
            )
            if model_slug in model_dir_text and has_weights and model_dir.joinpath("tokenizer_config.json").exists():
                return model_dir

    return Path(kagglehub.model_download(handle))


def gpu_memory_snapshot(label: str) -> None:
    allocated_gb = torch.cuda.memory_allocated() / (1024**3)
    reserved_gb = torch.cuda.memory_reserved() / (1024**3)
    print(f"{label}: allocated={allocated_gb:.2f} GB reserved={reserved_gb:.2f} GB")


def discover_tabular_files(root_dir: Path) -> List[Path]:
    supported_suffixes = {
        ".csv",
        ".json",
        ".jsonl",
        ".parquet",
        ".pq",
        ".tsv",
    }
    files = []
    for file_path in root_dir.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in supported_suffixes:
            files.append(file_path)
    return sorted(files)


def load_table(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".tsv":
        return pd.read_csv(file_path, sep="\t")
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(file_path)
    if suffix == ".jsonl":
        return pd.read_json(file_path, lines=True)
    if suffix == ".json":
        raw = json.loads(file_path.read_text())
        if isinstance(raw, list):
            return pd.DataFrame(raw)
        if isinstance(raw, dict):
            for value in raw.values():
                if isinstance(value, list):
                    return pd.DataFrame(value)
        raise ValueError(f"Unsupported JSON structure in {file_path}")
    raise ValueError(f"Unsupported file type: {file_path}")


def infer_column(columns: Iterable[str], candidates: Sequence[str]) -> Optional[str]:
    normalized_columns = {column.lower().strip(): column for column in columns}
    for candidate in candidates:
        if candidate in normalized_columns:
            return normalized_columns[candidate]

    for candidate in candidates:
        for normalized_name, original_name in normalized_columns.items():
            if candidate in normalized_name:
                return original_name

    return None


def to_supervised_frame(
    frame: pd.DataFrame,
    question_column: str,
    answer_column: str,
    source_name: str,
    file_name: str,
) -> pd.DataFrame:
    cleaned = frame[[question_column, answer_column]].copy()
    cleaned.columns = ["question", "answer"]
    cleaned = cleaned.dropna(subset=["question", "answer"])
    cleaned["question"] = cleaned["question"].astype(str).str.strip()
    cleaned["answer"] = cleaned["answer"].astype(str).str.strip()
    cleaned = cleaned[(cleaned["question"] != "") & (cleaned["answer"] != "")]
    cleaned["source"] = source_name
    cleaned["file_name"] = file_name
    cleaned = cleaned.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)
    return cleaned


def collect_supervised_candidates(root_dir: Path, source_name: str) -> List[Tuple[Path, pd.DataFrame]]:
    candidates: List[Tuple[Path, pd.DataFrame]] = []
    for file_path in discover_tabular_files(root_dir):
        try:
            frame = load_table(file_path)
        except Exception as exc:
            print(f"Skipping {file_path.name}: {exc}")
            continue

        question_column = infer_column(frame.columns, QUESTION_COLUMN_CANDIDATES)
        answer_column = infer_column(frame.columns, ANSWER_COLUMN_CANDIDATES)
        if question_column is None or answer_column is None:
            continue

        cleaned = to_supervised_frame(
            frame=frame,
            question_column=question_column,
            answer_column=answer_column,
            source_name=source_name,
            file_name=file_path.name,
        )
        if not cleaned.empty:
            candidates.append((file_path, cleaned))

    return candidates


def select_best_candidate(
    candidates: Sequence[Tuple[Path, pd.DataFrame]],
    split: str,
    exclude_paths: Optional[Sequence[Path]] = None,
) -> Optional[Tuple[Path, pd.DataFrame]]:
    if not candidates:
        return None

    exclude_set = {path.resolve() for path in (exclude_paths or [])}
    split_weights = {
        "train": {
            "train": 8,
            "main": 3,
            "socratic": 1,
            "valid": -4,
            "val": -4,
            "dev": -4,
            "test": -6,
        },
        "valid": {
            "valid": 8,
            "val": 8,
            "dev": 8,
            "test": 7,
            "main": 1,
            "socratic": 1,
            "train": -2,
        },
    }

    scored_candidates = []
    for file_path, frame in candidates:
        if file_path.resolve() in exclude_set:
            continue

        file_name = file_path.name.lower()
        score = min(len(frame), 100000)
        for token, weight in split_weights[split].items():
            if token in file_name:
                score += weight
        scored_candidates.append((score, file_path, frame))

    if not scored_candidates:
        return None

    scored_candidates.sort(key=lambda item: item[0], reverse=True)
    _, best_path, best_frame = scored_candidates[0]
    return best_path, best_frame


def sample_frame(frame: pd.DataFrame, max_rows: Optional[int], seed: int) -> pd.DataFrame:
    if max_rows is None or len(frame) <= max_rows:
        return frame.reset_index(drop=True)
    return frame.sample(n=max_rows, random_state=seed).reset_index(drop=True)


def split_train_valid(frame: pd.DataFrame, valid_rows: int, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if frame.empty:
        return frame.copy(), frame.copy()

    shuffled = frame.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    bounded_valid_rows = min(valid_rows, max(1, len(shuffled) - 1))
    valid_df = shuffled.iloc[:bounded_valid_rows].copy().reset_index(drop=True)
    train_df = shuffled.iloc[bounded_valid_rows:].copy().reset_index(drop=True)
    return train_df, valid_df


def summarize_candidates(label: str, candidates: Sequence[Tuple[Path, pd.DataFrame]]) -> None:
    print(f"\n{label} candidates:")
    if not candidates:
        print("  none")
        return
    for file_path, frame in candidates:
        print(f"  {file_path.name}: rows={len(frame)} columns={list(frame.columns)}")


def summarize_frame(label: str, frame: pd.DataFrame) -> None:
    print(f"\n{label}: rows={len(frame)}")
    if "source" in frame.columns:
        print(frame["source"].value_counts().to_string())
    display(frame.head(3))


def extract_boxed_answer(text: str) -> str:
    if not isinstance(text, str):
        return ""

    marker_match = FINAL_MARKER_PATTERN.search(text)
    if marker_match:
        return marker_match.group(1).strip().replace(",", "")

    matches = BOXED_PATTERN.findall(text)
    if matches:
        return matches[-1].strip()

    numeric_matches = NUMBER_PATTERN.findall(text)
    if numeric_matches:
        return numeric_matches[-1].replace(",", "").strip()

    return text.strip()


def normalize_answer_for_training(answer_text: str) -> str:
    raw_text = str(answer_text).strip()
    marker_match = FINAL_MARKER_PATTERN.search(raw_text)
    if marker_match:
        final_answer = marker_match.group(1).strip().replace(",", "")
        reasoning = raw_text[: marker_match.start()].strip()
        if reasoning:
            return f"{reasoning}\n\nTherefore the final answer is \\boxed{{{final_answer}}}."
        return f"The final answer is \\boxed{{{final_answer}}}."

    if "\\boxed{" in raw_text:
        return raw_text

    cleaned = extract_boxed_answer(raw_text)
    if cleaned:
        return f"The final answer is \\boxed{{{cleaned}}}."
    return raw_text


def maybe_parse_number(text: str) -> Optional[float]:
    cleaned = text.replace(",", "").strip()
    if cleaned.count("/") == 1 and all(part.strip("-").isdigit() for part in cleaned.split("/")):
        numerator, denominator = cleaned.split("/")
        denominator_value = float(denominator)
        if denominator_value == 0:
            return None
        return float(numerator) / denominator_value

    try:
        return float(cleaned)
    except ValueError:
        return None


def answers_match(predicted: str, target: str, tolerance: float = 1e-2) -> bool:
    normalized_prediction = extract_boxed_answer(predicted)
    normalized_target = extract_boxed_answer(target)
    if normalized_prediction == normalized_target:
        return True

    prediction_value = maybe_parse_number(normalized_prediction)
    target_value = maybe_parse_number(normalized_target)
    if prediction_value is None or target_value is None:
        return False

    if target_value == 0:
        return abs(prediction_value) <= tolerance

    relative_error = abs(prediction_value - target_value) / abs(target_value)
    return relative_error <= tolerance


def serialize_metrics(metrics: Dict[str, Any]) -> Dict[str, Any]:
    serialized = {}
    for key, value in metrics.items():
        if isinstance(value, (np.floating, float)):
            serialized[key] = float(value)
        elif isinstance(value, (np.integer, int)):
            serialized[key] = int(value)
        else:
            serialized[key] = str(value)
    return serialized


def render_chat(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        kwargs = {
            "tokenize": False,
            "add_generation_prompt": add_generation_prompt,
        }
        signature = inspect.signature(tokenizer.apply_chat_template)
        if "enable_thinking" in signature.parameters:
            kwargs["enable_thinking"] = False
        return tokenizer.apply_chat_template(messages, **kwargs)

    text_blocks = [f"{message['role'].upper()}: {message['content']}" for message in messages]
    if add_generation_prompt:
        text_blocks.append("ASSISTANT:")
    return "\n\n".join(text_blocks)


def tokenize_supervised_examples(
    tokenizer,
    frame: pd.DataFrame,
    max_length: int,
) -> Dataset:
    records = []
    for row in frame.itertuples(index=False):
        question = str(row.question)
        answer = normalize_answer_for_training(str(row.answer))
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
        full_messages = prompt_messages + [{"role": "assistant", "content": answer}]

        prompt_text = render_chat(tokenizer, prompt_messages, add_generation_prompt=True)
        full_text = render_chat(tokenizer, full_messages, add_generation_prompt=False)

        prompt_tokens = tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
        )
        full_tokens = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
        )

        labels = list(full_tokens["input_ids"])
        prompt_token_count = min(len(prompt_tokens["input_ids"]), len(labels))
        labels[:prompt_token_count] = [-100] * prompt_token_count

        if all(label == -100 for label in labels):
            continue

        records.append(
            {
                "input_ids": full_tokens["input_ids"],
                "attention_mask": full_tokens["attention_mask"],
                "labels": labels,
            }
        )

    return Dataset.from_list(records)


def build_generation_inputs(tokenizer, question: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt_text = render_chat(tokenizer, messages, add_generation_prompt=True)
    return tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)




In [ ]:
competition_dir = resolve_competition_dir(COMPETITION_HANDLE)
aux_dataset_dir = resolve_dataset_dir(AUX_DATASET_HANDLE)

competition_candidates = collect_supervised_candidates(competition_dir, source_name="competition")
aux_candidates = collect_supervised_candidates(aux_dataset_dir, source_name=handle_slug(AUX_DATASET_HANDLE))

if VARIANT["debug"]:
    summarize_candidates("competition", competition_candidates)
    summarize_candidates("auxiliary", aux_candidates)

competition_train_candidate = select_best_candidate(competition_candidates, split="train")
competition_valid_candidate = select_best_candidate(
    competition_candidates,
    split="valid",
    exclude_paths=[competition_train_candidate[0]] if competition_train_candidate else None,
)
aux_train_candidate = select_best_candidate(aux_candidates, split="train")
aux_valid_candidate = select_best_candidate(
    aux_candidates,
    split="valid",
    exclude_paths=[aux_train_candidate[0]] if aux_train_candidate else None,
)

competition_train_df = (
    competition_train_candidate[1].copy()
    if competition_train_candidate is not None
    else pd.DataFrame(columns=["question", "answer", "source", "file_name"])
)
competition_valid_df = (
    competition_valid_candidate[1].copy()
    if competition_valid_candidate is not None
    else pd.DataFrame(columns=["question", "answer", "source", "file_name"])
)
aux_train_df = (
    aux_train_candidate[1].copy()
    if aux_train_candidate is not None
    else pd.DataFrame(columns=["question", "answer", "source", "file_name"])
)
aux_valid_df = (
    aux_valid_candidate[1].copy()
    if aux_valid_candidate is not None
    else pd.DataFrame(columns=["question", "answer", "source", "file_name"])
)

if competition_train_df.empty and aux_train_df.empty:
    raise RuntimeError(
        "No labeled training data was found in either the competition files or the auxiliary Kaggle dataset."
    )

train_parts = []
if not competition_train_df.empty:
    train_parts.append(
        sample_frame(
            competition_train_df,
            max_rows=int(VARIANT["max_competition_train_samples"]),
            seed=SEED,
        )
    )
if VARIANT["use_aux_dataset"] and not aux_train_df.empty:
    train_parts.append(
        sample_frame(
            aux_train_df,
            max_rows=int(VARIANT["max_aux_train_samples"]),
            seed=SEED + 1,
        )
    )

train_df = pd.concat(train_parts, ignore_index=True)
train_df = train_df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

valid_parts = []
if not competition_valid_df.empty:
    valid_parts.append(sample_frame(competition_valid_df, max_rows=MAX_VALID_SAMPLES, seed=SEED + 2))
if VARIANT["use_aux_dataset"] and not aux_valid_df.empty:
    valid_parts.append(sample_frame(aux_valid_df, max_rows=MAX_VALID_SAMPLES, seed=SEED + 3))

if valid_parts:
    valid_df = pd.concat(valid_parts, ignore_index=True)
    valid_df = valid_df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)
    valid_df = sample_frame(valid_df, max_rows=MAX_VALID_SAMPLES, seed=SEED + 4)
else:
    train_df, valid_df = split_train_valid(train_df, valid_rows=MAX_VALID_SAMPLES, seed=SEED + 5)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

if train_df.empty:
    raise RuntimeError("Training data became empty after preprocessing and sampling.")

if valid_df.empty:
    train_df, valid_df = split_train_valid(train_df, valid_rows=min(MAX_VALID_SAMPLES, 8), seed=SEED + 6)

print(f"Competition directory: {competition_dir}")
print(f"Auxiliary dataset directory: {aux_dataset_dir}")
summarize_frame("Training frame", train_df)
summarize_frame("Validation frame", valid_df)



In [ ]:
model_dir = resolve_model_dir(MODEL_HANDLE)
print(f"Model directory: {model_dir}")
gpu_memory_snapshot("Before model load")

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=True,
        device_map="auto",
        dtype=COMPUTE_DTYPE,
        quantization_config=quantization_config,
    )
except ImportError as exc:
    error_text = str(exc).lower()
    if any(token in error_text for token in ("mamba-ssm", "mamba_ssm", "causal-conv1d", "causal_conv1d")):
        raise RuntimeError(
            "Nemotron custom model code could not import its Mamba runtime. "
            "Cell 1 installs mamba-ssm and causal-conv1d automatically. "
            "Restart the notebook kernel and rerun from Cell 1. "
            "If Kaggle assigned a P100, restart the Kaggle session until you get a newer GPU first."
        ) from exc
    raise
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
gpu_memory_snapshot("After model load")

train_dataset = tokenize_supervised_examples(tokenizer, train_df, max_length=MAX_SEQ_LENGTH)
valid_dataset = tokenize_supervised_examples(tokenizer, valid_df, max_length=MAX_SEQ_LENGTH)

if len(train_dataset) == 0:
    raise RuntimeError("Tokenization produced zero training rows. Reduce MAX_SEQ_LENGTH or inspect the source columns.")

if len(valid_dataset) == 0:
    valid_dataset = train_dataset.select(range(min(8, len(train_dataset))))

print(f"Tokenized training rows: {len(train_dataset)}")
print(f"Tokenized validation rows: {len(valid_dataset)}")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(RUN_DIR),
    overwrite_output_dir=True,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_strategy="steps",
    logging_steps=int(VARIANT["logging_steps"]),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    report_to="none",
    bf16=BF16_AVAILABLE,
    fp16=not BF16_AVAILABLE,
    save_only_model=True,
    run_name=RUN_NAME,
    log_level="info" if VARIANT["debug"] else "warning",
    skip_memory_metrics=not VARIANT["debug"],
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    eval_on_start=bool(VARIANT["debug"]),
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
)



In [ ]:
gpu_memory_snapshot("Before training")
train_result = trainer.train()
gpu_memory_snapshot("After training")
print(json.dumps(serialize_metrics(train_result.metrics), indent=2))



In [ ]:
preview_rows = []
model.eval()
inference_device = next(model.parameters()).device

for row in valid_df.head(4).itertuples(index=False):
    generation_inputs = build_generation_inputs(tokenizer, str(row.question))
    generation_inputs = {key: value.to(inference_device) for key, value in generation_inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **generation_inputs,
            max_new_tokens=int(VARIANT["preview_max_new_tokens"]),
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = generation_inputs["input_ids"].shape[1]
    decoded_text = tokenizer.decode(generated[0][prompt_length:], skip_special_tokens=True)
    preview_rows.append(
        {
            "question": str(row.question)[:140],
            "target": extract_boxed_answer(str(row.answer)),
            "prediction": extract_boxed_answer(decoded_text),
            "match": answers_match(decoded_text, str(row.answer)),
            "source": getattr(row, "source", "unknown"),
        }
    )

preview_df = pd.DataFrame(preview_rows)
display(preview_df)



In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
preview_df.to_csv(ARTIFACT_DIR / "preview_predictions.csv", index=False)

summary = {
    "run_name": RUN_NAME,
    "active_variant": ACTIVE_VARIANT,
    "variant_config": VARIANT,
    "competition_handle": COMPETITION_HANDLE,
    "aux_dataset_handle": AUX_DATASET_HANDLE,
    "model_handle": MODEL_HANDLE,
    "train_rows": len(train_df),
    "valid_rows": len(valid_df),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_rank": LORA_RANK,
    "epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_metrics": serialize_metrics(train_result.metrics),
    "train_sources": train_df["source"].value_counts().to_dict() if "source" in train_df.columns else {},
    "valid_sources": valid_df["source"].value_counts().to_dict() if "source" in valid_df.columns else {},
}
(ARTIFACT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2))

if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(ADAPTER_DIR.rglob("*")):
        if file_path.is_file():
            archive.write(file_path, arcname=str(file_path.relative_to(ADAPTER_DIR)))

adapter_files = sorted(path.name for path in ADAPTER_DIR.iterdir() if path.is_file())
print(f"Adapter files: {adapter_files}")
print(f"Submission ready: {SUBMISSION_PATH}")
